In [16]:
import pandas as pd
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, sessionmaker

# 创建数据库连接
Base = declarative_base()
engine = create_engine('sqlite:///database.db')
Session = sessionmaker(bind=engine)
session = Session()

# 定义 SpatialLayer 表
class SpatialLayer(Base):
    __tablename__ = 'spatiallayer'
    id = Column(Integer, primary_key=True, autoincrement=True)
    SLID = Column(String, unique=True, nullable=False)  # 唯一标识符
    species = Column(String)
    tissue_class = Column(String)
    tissue_type = Column(String)
    uberonongology_id=Column(String)
    cancer_type = Column(String)
    spatial_layer = Column(String)
    Cell_type_composition = Column(String)
    GeneID = Column(String)
    Genetype = Column(String)
    Genename = Column(String)
    UNIPROTID = Column(String)
    PMID = Column(String)
    Paper_Title = Column(String)
    journal = Column(String)
    year = Column(String)
    technology_type_for_discovery = Column(String)
    technology_platform_for_discovery = Column(String)
    Phenotype_type = Column(String)
    Phenotype_label = Column(String)
    model_type = Column(String)
    technology_type_for_validation = Column(String)
    technology_platform_for_validation = Column(String)
    evidence_type = Column(String)
    Phenotype_evidence = Column(String)

# 定义 CellType 表
class CellType(Base):
    __tablename__ = 'celltype'
    id = Column(Integer, primary_key=True, autoincrement=True)
    CTID = Column(String, unique=True, nullable=False)  # 唯一标识符
    species = Column(String)
    tissue_class = Column(String)
    tissue_type = Column(String)
    uberonongology_id=Column(String)
    cancer_type = Column(String)
    major_cell_type = Column(String)
    cell_name = Column(String)
    cellontology_id = Column(String)
    marker = Column(String)
    Symbol = Column(String)
    GeneID = Column(String)
    Genetype = Column(String)
    Genename = Column(String)
    UNIPROTID = Column(String)
    PMID = Column(String)
    Paper_Title = Column(String)
    journal = Column(String)
    year = Column(String)
    technology_type_for_discovery = Column(String)
    technology_platform_for_discovery = Column(String)
    Phenotype_type = Column(String)
    Phenotype_label = Column(String)
    model_type = Column(String)
    technology_type_for_validation = Column(String)
    technology_platform_for_validation = Column(String)
    evidence_type = Column(String)
    Phenotype_evidence = Column(String)
    curator=Column(String)


# 创建表结构
Base.metadata.create_all(engine)

# 定义字段分类规则
SPATIALLAYER_FIELDS = {
    "basic_info": ["species", "tissue_class", "tissue_type", "cancer_type"],
    "spatial_info": ["SLID", "spatial_layer", "Cell_type_composition"],
    "paper_info": ["PMID", "Paper_Title", "journal", "year"],
    "technology_info": [
        "technology_type_for_discovery",
        "technology_platform_for_discovery",
        "technology_type_for_validation",
        "technology_platform_for_validation",
        "model_type",
        "evidence_type"
    ],
    "phenotype_info": ["Phenotype_type", "Phenotype_label", "Phenotype_evidence"],
}

CELLTYPE_FIELDS = {
    "basic_info": ["species", "tissue_class", "tissue_type", "cancer_type"],
    "cell_info": ["CTID", "major_cell_type", "cell_name", "Symbol"],
    "paper_info": ["PMID", "Paper_Title", "journal", "year"],
    "technology_info": [
        "technology_type_for_discovery",
        "technology_platform_for_discovery",
        "technology_type_for_validation",
        "technology_platform_for_validation",
        "model_type",
        "evidence_type"
    ],
    "phenotype_info": ["Phenotype_type", "Phenotype_label", "Phenotype_evidence"],
}

# 通用数据加载函数
def load_data_to_table(csv_path, table_class, unique_field):
    """
    从 CSV 文件加载数据到指定的数据库表
    """
    data = pd.read_csv(csv_path)

    # 检查唯一字段是否唯一
    if data[unique_field].duplicated().any():
        raise ValueError(f"{unique_field} 列包含重复值，请确保其是唯一的。")

    # 插入数据到指定表
    records = [
        table_class(**row.to_dict()) for _, row in data.iterrows()
    ]

    # 批量插入数据库
    session.bulk_save_objects(records)
    session.commit()
    print(f"数据已成功加载到 {table_class.__tablename__} 表中，共加载 {len(records)} 条记录。")

# 通用数据查询函数
def format_entry(entry, category_fields):
    """
    按类别分组数据
    """
    formatted_entry = {}
    for category, fields in category_fields.items():
        formatted_entry[category] = {field: getattr(entry, field, None) for field in fields}
    return formatted_entry

def get_entry_by_unique_field(table_class, field_name, field_value, category_fields):
    """
    根据唯一字段查询数据表中的数据并按分类返回
    """
    entry = session.query(table_class).filter_by(**{field_name: field_value}).first()
    if not entry:
        return None
    return format_entry(entry, category_fields)

# 加载 SpatialLayer 数据
spatial_csv_path = '/mnt/data/ljj/Project_TiPhD/111/data/table_for_TiPhD_SpatiaNich.csv'
load_data_to_table(spatial_csv_path, SpatialLayer, 'SLID')

# 加载 CellType 数据
cell_csv_path = '/mnt/data/ljj/Project_TiPhD/111/data/data_table_3.17.csv'
load_data_to_table(cell_csv_path, CellType, 'CTID')

# 示例查询 SpatialLayer
SLID_example = "SL001"
spatial_entry = get_entry_by_unique_field(SpatialLayer, 'SLID', SLID_example, SPATIALLAYER_FIELDS)
if spatial_entry:
    print("SpatialLayer 表数据：")
    print(spatial_entry)
else:
    print(f"未找到 SLID 为 {SLID_example} 的条目。")

# 示例查询 CellType
CTID_example = "CT001"
cell_entry = get_entry_by_unique_field(CellType, 'CTID', CTID_example, CELLTYPE_FIELDS)
if cell_entry:
    print("CellType 表数据：")
    print(cell_entry)
else:
    print(f"未找到 CTID 为 {CTID_example} 的条目。")


数据已成功加载到 spatiallayer 表中，共加载 75 条记录。
数据已成功加载到 celltype 表中，共加载 300 条记录。
SpatialLayer 表数据：
{'basic_info': {'species': 'Human', 'tissue_class': 'Gut', 'tissue_type': 'Colorectum', 'cancer_type': 'Colorectal Cancer'}, 'spatial_info': {'SLID': 'SL001', 'spatial_layer': 'Desmoplastic structure formation', 'Cell_type_composition': 'FAP+ fibroblasts, SPP1+ macrophages'}, 'paper_info': {'PMID': '35365629.0', 'Paper_Title': 'Single-cell and spatial analysis reveal interaction of FAP+ fibroblasts and SPP1+ macrophages in colorectal cancer', 'journal': 'Nat Commun', 'year': '2022'}, 'technology_info': {'technology_type_for_discovery': 'spatial transcriptome', 'technology_platform_for_discovery': '10x Genomics', 'technology_type_for_validation': 'H-score system in tissue microarray', 'technology_platform_for_validation': None, 'model_type': 'clinical samples', 'evidence_type': 'clinical data'}, 'phenotype_info': {'Phenotype_type': 'Clinical phenotype', 'Phenotype_label': 'Shorter survival', 'Phenot

In [1]:
import pandas as pd
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

# 创建数据库连接
Base = declarative_base()
engine = create_engine('sqlite:///database.db')
Session = sessionmaker(bind=engine)
session = Session()

# 定义 SpatialLayer 表
class SpatialLayer(Base):
    __tablename__ = 'spatiallayer'
    id = Column(Integer, primary_key=True, autoincrement=True)
    SLID = Column(String, unique=True, nullable=False)  # 唯一标识符
    species = Column(String)
    tissue_class = Column(String)
    tissue_type = Column(String)
    cancer_type = Column(String)
    spatial_layer = Column(String)
    cell_type_composition = Column(String)
    GeneID = Column(String)
    Genetype = Column(String)
    Genename = Column(String)
    UNIPROTID = Column(String)
    PMID = Column(String)
    Paper_Title = Column(String)
    journal = Column(String)
    year = Column(String)
    technology_type_for_discovery = Column(String)
    technology_platform_for_discovery = Column(String)
    Phenotype_type = Column(String)
    Phenotype_label = Column(String)
    model_type = Column(String)
    technology_type_for_validation = Column(String)
    technology_platform_for_validation = Column(String)
    evidence_type = Column(String)
    Phenotype_evidence = Column(String)

# 创建表结构
Base.metadata.create_all(engine)

# 定义字段分类规则
SPATIALLAYER_FIELDS = {
    "basic_info": ["species", "tissue_class", "tissue_type", "cancer_type"],
    "spatial_info": ["SLID", "spatial_layer", "Cell_type_composition"],
    "paper_info": ["PMID", "Paper_Title", "journal", "year"],
    "technology_info": [
        "technology_type_for_discovery",
        "technology_platform_for_discovery",
        "technology_type_for_validation",
        "technology_platform_for_validation",
        "model_type",
        "evidence_type"
    ],
    "phenotype_info": ["Phenotype_type", "Phenotype_label", "Phenotype_evidence"],
}

def format_entry(entry, category_fields):
    """
    按类别分组数据
    """
    formatted_entry = {}
    for category, fields in category_fields.items():
        formatted_entry[category] = {field: getattr(entry, field, None) for field in fields}
    return formatted_entry

# 加载数据
def load_data(csv_path):
    """
    从 CSV 文件加载数据到数据库
    """
    data = pd.read_csv(csv_path)

    # 检查 SLID 是否唯一
    if data['SLID'].duplicated().any():
        raise ValueError("SLID 列包含重复值，请确保 SLID 是唯一的。")

    # 插入数据到 SpatialLayer 表
    records = []
    for _, row in data.iterrows():
        record = SpatialLayer(
            SLID=row.get('SLID'),
            species=row.get('species'),
            tissue_class=row.get('tissue_class'),
            tissue_type=row.get('tissue_type'),
            cancer_type=row.get('cancer_type'),
            spatial_layer = row.get('spatial_layer'),
            cell_type_composition = row.get('Cell_type_composition'),
            GeneID=row.get('GeneID'),
            Genetype=row.get('Genetype'),
            Genename=row.get('Genename'),
            UNIPROTID=row.get('UNIPROTID'),
            PMID=row.get('PMID'),
            Paper_Title=row.get('Paper_Title'),
            journal=row.get('journal'),
            year=row.get('year'),
            technology_type_for_discovery=row.get('technology_type_for_discovery'),
            technology_platform_for_discovery=row.get('technology_platform_for_discovery'),
            Phenotype_type=row.get('Phenotype_type'),
            Phenotype_label=row.get('Phenotype_label'),
            model_type=row.get('model_type'),
            technology_type_for_validation=row.get('technology_type_for_validation'),
            technology_platform_for_validation=row.get('technology_platform_for_validation'),
            evidence_type=row.get('evidence_type'),
            Phenotype_evidence=row.get('Phenotype_evidence')
        )
        records.append(record)

    # 批量插入数据库
    session.bulk_save_objects(records)
    session.commit()
    print(f"数据已成功加载到 SpatialLayer 表中，共加载 {len(records)} 条记录。")

# 示例：从 CSV 文件加载数据
csv_path = '/mnt/data/ljj/Project_TiPhD/111/data/table_for_TiPhD_SpatiaNich.csv'
load_data(csv_path)

# 查询并格式化数据
def get_entry_by_SLID(SLID):
    """
    根据 SLID 查询 SpatialLayer 数据并按分类返回
    """
    entry = session.query(SpatialLayer).filter_by(SLID=SLID).first()
    if not entry:
        return None
    return format_entry(entry, SPATIALLAYER_FIELDS)




数据已成功加载到 SpatialLayer 表中，共加载 75 条记录。


In [3]:
# 示例查询
SLID_example = "SL001"  # 替换为实际 SLID
entry_data = get_entry_by_SLID(SLID_example)
if entry_data:
    print("按类别分组的数据：")
    print(entry_data)
else:
    print(f"未找到 SLID 为 {SLID_example} 的条目。")

按类别分组的数据：
{'basic_info': {'species': 'Human', 'tissue_class': 'Gut', 'tissue_type': 'Colorectum', 'cancer_type': 'Colorectal Cancer'}, 'spatial_info': {'SLID': 'SL001', 'spatial_layer': 'Desmoplastic structure formation', 'Cell_type_composition': None}, 'paper_info': {'PMID': '35365629.0', 'Paper_Title': 'Single-cell and spatial analysis reveal interaction of FAP+ fibroblasts and SPP1+ macrophages in colorectal cancer', 'journal': 'Nat Commun', 'year': '2022'}, 'technology_info': {'technology_type_for_discovery': 'spatial transcriptome', 'technology_platform_for_discovery': '10x Genomics', 'technology_type_for_validation': 'H-score system in tissue microarray', 'technology_platform_for_validation': None, 'model_type': 'clinical samples', 'evidence_type': 'clinical data'}, 'phenotype_info': {'Phenotype_type': 'Clinical phenotype', 'Phenotype_label': 'Shorter survival', 'Phenotype_evidence': 'We found that the FAP+ fibroblasts and SPP1+ macrophages were the most highly correlated populat

In [4]:
import pandas as pd
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

# 创建数据库连接
Base = declarative_base()
engine = create_engine('sqlite:///database.db')
Session = sessionmaker(bind=engine)
session = Session()

# 定义 CellType 表
class CellType(Base):
    __tablename__ = 'celltype'
    id = Column(Integer, primary_key=True, autoincrement=True)
    CTID = Column(String, unique=True, nullable=False)  # 唯一标识符
    species = Column(String)
    tissue_class = Column(String)
    tissue_type = Column(String)
    cancer_type = Column(String)
    major_cell_type = Column(String)
    cell_name = Column(String)
    cellontology_id = Column(String)
    marker = Column(String)
    Symbol = Column(String)
    GeneID = Column(String)
    Genetype = Column(String)
    Genename = Column(String)
    UNIPROTID = Column(String)
    PMID = Column(String)
    Paper_Title = Column(String)
    journal = Column(String)
    year = Column(String)
    technology_type_for_discovery = Column(String)
    technology_platform_for_discovery = Column(String)
    Phenotype_type = Column(String)
    Phenotype_label = Column(String)
    model_type = Column(String)
    technology_type_for_validation = Column(String)
    technology_platform_for_validation = Column(String)
    evidence_type = Column(String)
    Phenotype_evidence = Column(String)

# 创建表结构
Base.metadata.create_all(engine)

# 定义字段分类规则
CELLTYPE_FIELDS = {
    "basic_info": ["species", "tissue_class", "tissue_type", "cancer_type"],
    "cell_info": ["CTID", "major_cell_type", "cell_name","Symbol"],
    "paper_info": ["PMID", "Paper_Title", "journal", "year"],
    "technology_info": [
        "technology_type_for_discovery",
        "technology_platform_for_discovery",
        "technology_type_for_validation",
        "technology_platform_for_validation",
        "model_type",
        "evidence_type"
    ],
    "phenotype_info": ["Phenotype_type", "Phenotype_label", "Phenotype_evidence"],
}

def format_entry(entry, category_fields):
    """
    按类别分组数据
    """
    formatted_entry = {}
    for category, fields in category_fields.items():
        formatted_entry[category] = {field: getattr(entry, field, None) for field in fields}
    return formatted_entry

# 加载数据
def load_data(csv_path):
    """
    从 CSV 文件加载数据到数据库
    """
    data = pd.read_csv(csv_path)

    # 检查 CTID 是否唯一
    if data['CTID'].duplicated().any():
        raise ValueError("CTID 列包含重复值，请确保 CTID 是唯一的。")

    # 插入数据到 CellType 表
    records = []
    for _, row in data.iterrows():
        record = CellType(
            CTID=row.get('CTID'),
            species=row.get('species'),
            tissue_class=row.get('tissue_class'),
            tissue_type=row.get('tissue_type'),
            cancer_type=row.get('cancer_type'),
            major_cell_type=row.get('major_cell_type'),
            cell_name=row.get('cell_name'),
            cellontology_id=row.get('cellontology_id'),
            marker=row.get('marker'),
            Symbol=row.get('Symbol'),
            GeneID=row.get('GeneID'),
            Genetype=row.get('Genetype'),
            Genename=row.get('Genename'),
            UNIPROTID=row.get('UNIPROTID'),
            PMID=row.get('PMID'),
            Paper_Title=row.get('Paper_Title'),
            journal=row.get('journal'),
            year=row.get('year'),
            technology_type_for_discovery=row.get('technology_type_for_discovery'),
            technology_platform_for_discovery=row.get('technology_platform_for_discovery'),
            Phenotype_type=row.get('Phenotype_type'),
            Phenotype_label=row.get('Phenotype_label'),
            model_type=row.get('model_type'),
            technology_type_for_validation=row.get('technology_type_for_validation'),
            technology_platform_for_validation=row.get('technology_platform_for_validation'),
            evidence_type=row.get('evidence_type'),
            Phenotype_evidence=row.get('Phenotype_evidence')
        )
        records.append(record)

    # 批量插入数据库
    session.bulk_save_objects(records)
    session.commit()
    print(f"数据已成功加载到 CellType 表中，共加载 {len(records)} 条记录。")

# 示例：从 CSV 文件加载数据
csv_path = '/mnt/data/ljj/Project_TiPhD/111/data/data_table_3.17.csv'
load_data(csv_path)

# 查询并格式化数据
def get_entry_by_CTID(CTID):
    """
    根据 CTID 查询 CellType 数据并按分类返回
    """
    entry = session.query(CellType).filter_by(CTID=CTID).first()
    if not entry:
        return None
    return format_entry(entry, CELLTYPE_FIELDS)




数据已成功加载到 CellType 表中，共加载 300 条记录。


In [6]:
# 示例查询
CTID_example = "CT001"  # 替换为实际 SLID
entry_data = get_entry_by_CTID(CTID_example)
if entry_data:
    print("按类别分组的数据：")
    print(entry_data)
else:
    print(f"未找到 SLID 为 {CTID_example} 的条目。")

按类别分组的数据：
{'basic_info': {'species': 'Human', 'tissue_class': 'Liver', 'tissue_type': 'Liver', 'cancer_type': 'Hepatocelluar carcinoma'}, 'cell_info': {'CTID': 'CT001', 'major_cell_type': 'Macrophages', 'cell_name': 'SPP1+ macrophages', 'Symbol': 'SPP1,CD68,LUM,TIMP1'}, 'paper_info': {'PMID': '36708811', 'Paper_Title': 'Identification of a tumour immune barrier in the HCC microenvironment that determines the efficacy of immunotherapy', 'journal': 'Journal of hepatology', 'year': '2023'}, 'technology_info': {'technology_type_for_discovery': 'spatial transcriptome', 'technology_platform_for_discovery': '10 × Spatial', 'technology_type_for_validation': 'survival analysis', 'technology_platform_for_validation': 'analysis', 'model_type': 'clinical samples', 'evidence_type': 'clinical data'}, 'phenotype_info': {'Phenotype_type': 'Clinical phenotype', 'Phenotype_label': 'shorter survival', 'Phenotype_evidence': 'Notably, patients with HCC and higher infiltration of SPP1+ macrophages had short